# Домашняя работа 8. K-средних и EM своими руками

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 8 — Кластеризация и EM-алгоритм |
| Опора | материал семинара 8 и лекций до него |
| Ожидаемое время | 3–4 часа |

Оба алгоритма занятия — итеративные схемы из двух шагов, и написать их проще, чем кажется. Зато станет видно, чем они на самом деле различаются: жёсткое отнесение против мягкого — это одна строка кода.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=8)
describe_variant(variant)

In [ ]:
X_b, y_b = make_blobs(n_samples=500, centers=4, cluster_std=1.1, random_state=RANDOM_STATE)
X_b = StandardScaler().fit_transform(X_b)
print(f"выборка: {X_b.shape}, истинных кластеров: {len(np.unique(y_b))}")

---
# Задача 1. Алгоритм Ллойда и k-means++

Два шага, каждый из которых решает свою подзадачу точно:

1. при фиксированных центрах отнести каждый объект к ближайшему;
2. при фиксированном разбиении пересчитать центры как средние кластеров.

Инициализация `k-means++`: первый центр случаен, каждый следующий выбирается
с вероятностью, пропорциональной **квадрату** расстояния до ближайшего из уже
выбранных.

In [ ]:
def kmeans_pp_init(X, K, generator):
    """k-means++: первый центр случаен; каждый следующий выбирается
    с вероятностью, пропорциональной КВАДРАТУ расстояния до ближайшего
    из уже выбранных."""
    raise NotImplementedError


def kmeans(X, K, init="++", n_iter=200, tol=1e-10, seed=0):
    """Алгоритм Ллойда. Возвращает (метки, центры, история Q).

    Шаг 1: d2 = квадраты расстояний до центров (n, K); labels = argmin по оси 1;
           Q -- сумма минимальных расстояний.
    Шаг 2: новые центры -- средние по кластерам (пустой кластер оставить как был).
    Остановка: центры сдвинулись меньше чем на tol.
    """
    raise NotImplementedError

In [ ]:
# TODO: 1) сверьте своё Q и разбиение со sklearn.cluster.KMeans (ARI ~ 1.0);
#       2) проверьте, что история Q убывает монотонно;
#       3) постройте график Q по итерациям.

> **Вывод.** Убывает ли $Q$ монотонно и почему это гарантировано? Сколько итераций потребовалось?
>
> *(ваш ответ здесь)*

---
# Задача 2. EM для смеси гауссиан

**E-шаг**: $\gamma_{ik} = \dfrac{w_k\,\mathcal N(x_i \mid \mu_k, \Sigma_k)}
{\sum_s w_s\,\mathcal N(x_i \mid \mu_s, \Sigma_s)}$.

**M-шаг**: $N_k = \sum_i\gamma_{ik}$, и далее
$w_k = N_k/\ell$, $\mu_k = \frac{1}{N_k}\sum_i\gamma_{ik}x_i$,
$\Sigma_k = \frac{1}{N_k}\sum_i\gamma_{ik}(x_i-\mu_k)(x_i-\mu_k)^{\mathsf T}$.

Важно считать E-шаг **в логарифмах**: произведение плотностей в многомерном
случае легко обнуляется, и деление даёт `nan`. Приём тот же, что с сигмоидой
в занятии 4: вычесть максимум перед экспонированием.

In [ ]:
def gmm_em(X, K, cov_type="full", n_iter=200, tol=1e-8, seed=0, reg=1e-6):
    """EM для смеси гауссиан. Возвращает (w, mu, Sigma, gamma, история logL).

    Инициализация: центры -- kmeans_pp_init, w = 1/K, Sigma = ковариация всей
    выборки (плюс reg * I для устойчивости).

    E-шаг: logp[:, k] = log w_k + multivariate_normal(mu_k, Sigma_k).logpdf(X);
        нормировка в логарифмах (вычесть максимум по строке, затем logsumexp);
        gamma = exp(logp - log_norm), а сумма log_norm -- это ln L.
    M-шаг: N_k = sum_i gamma_ik; w, mu, Sigma -- взвешенные оценки по формулам.
        Для 'diag' оставить только диагональ Sigma, для 'spherical' --
        единичную матрицу, умноженную на среднюю дисперсию (trace / d).
    """
    raise NotImplementedError

In [ ]:
X_g, _ = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
X_g = X_g @ np.array([[0.6, -0.6], [-0.4, 0.8]])
X_g = StandardScaler().fit_transform(X_g)

# TODO: обучите свой EM с типом ковариации из variant["gmm_cov"];
#   проверьте монотонность logL (теорема 7.7) и сверьтесь
#   с GaussianMixture по logL на объект и по ARI разбиений.

In [ ]:
# TODO: постройте два графика: (а) logL по итерациям; (б) кластеры с эллипсами
#       уровня ковариаций, прозрачность точки = max_k gamma_ik.

> **Вывод.** Совпало ли с `sklearn`? Что даёт мягкое отнесение по сравнению с $K$-средних и где в коде проходит граница между ними?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. EM гарантированно не уменьшает $\ln L$, но может сойтись к плохому локальному максимуму. Как с этим борются на практике?
2. Почему E-шаг обязательно считать в логарифмах? Что произойдёт при размерности 50, если считать плотности напрямую?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.